# Part 7 — Hybrid Retrieval

This notebook demonstrates the hybrid three-channel retrieval system (EP-12, EP-13):

| File | Responsibility |
|------|----------------|
| `src/retrieval/embeddings.py` | Singleton BGE-M3 encoder (1024-dim dense vectors) |
| `src/retrieval/hybrid_retriever.py` | Dense vector + BM25 + Graph traversal + merge |
| `src/retrieval/reranker.py` | Cross-encoder reranker (bge-reranker-large) |

**Pipeline overview:**
```
Query (natural language)
    │
    ▼
embed_text(query)           [BGE-M3 1024-dim vector]
    │
    ├── vector_search()     [Neo4j vector index — semantic similarity]
    ├── bm25_search()       [BM25Okapi — exact keyword matching]
    └── graph_traversal()   [Cypher hop expansion — relational context]
    │
    ▼
merge_results()             [Dedup by node_id, keep max score]
    │
    ▼
rerank()                    [bge-reranker-large cross-encoder]
    │
    ▼
list[RetrievedChunk]        [Top-K most relevant, sorted by score]
```

> **Note**: Real model calls (BGE-M3, bge-reranker-large) and Neo4j are mocked in this notebook so it can run fully offline.


In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /home/marcantoniolopez/Documenti/github/thesis


## 1. `embed_texts()` and `embed_text()` — BGE-M3 Dense Embeddings

`embed_texts(texts, model=None)` encodes a list of strings into 1024-dimensional vectors.
When `model=None` it calls the singleton `get_embeddings()`.
`embed_text(text, model=None)` is a convenience wrapper for a single string.

The 1024 dimension is a hard requirement: the Neo4j vector index is created with `vector.dimensions: 1024`.


In [2]:
import numpy as np
from unittest.mock import MagicMock

from src.retrieval.embeddings import embed_texts, embed_text

# Mock del FlagModel — evita il download di BAAI/bge-m3 in CI
def _fake_model(dim=1024):
    model = MagicMock()
    model.encode = MagicMock(
        side_effect=lambda texts, **kw: np.zeros((len(texts), dim), dtype="float32")
    )
    return model

fake_model = _fake_model()

# Batch encoding
texts = ["What is the Customer table?", "DDL source for TB_CUST", "purchase order"]
vectors = embed_texts(texts, model=fake_model)
print(f"Input texts:    {len(texts)}")
print(f"Output vectors: {len(vectors)}")
print(f"Vector dim:     {len(vectors[0])}")
assert len(vectors) == 3
assert len(vectors[0]) == 1024
print("✓ embed_texts — batch encoding OK")

Input texts:    3
Output vectors: 3
Vector dim:     1024
✓ embed_texts — batch encoding OK


/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')


In [3]:
# Single-text convenience wrapper
vec = embed_text("customer identifier", model=fake_model)
print(f"Single vector dim: {len(vec)}")
assert isinstance(vec, list)
assert len(vec) == 1024
print("✓ embed_text — single encoding OK")

Single vector dim: 1024
✓ embed_text — single encoding OK


In [4]:
# Empty input guard
result = embed_texts([], model=MagicMock())
assert result == []
print("✓ embed_texts([]) returns []")

✓ embed_texts([]) returns []


## 2. `_node_to_text()` — Node Flattening for BM25

`_node_to_text(node)` flattens a node dict into a searchable string for BM25,
concatenating `name`, `definition`, `synonyms` and `column_names`, all lowercased.


In [5]:
from src.retrieval.hybrid_retriever import _node_to_text

node = {
    "name": "Customer",
    "definition": "A person who purchases goods or services",
    "synonyms": ["Client", "Buyer"],
    "column_names": ["CUST_ID", "CUST_NM", "EMAIL"],
    "node_type": "BusinessConcept",
}
text = _node_to_text(node)
print(f"Flattened text: '{text}'")
assert "customer" in text
assert "client" in text
assert "buyer" in text
assert "cust_id" in text
print("✓ _node_to_text — includes all fields lowercased")

Flattened text: 'customer a person who purchases goods or services client buyer cust_id cust_nm email'
✓ _node_to_text — includes all fields lowercased


## 3. `bm25_search()` — BM25 Keyword Search

`bm25_search(query, all_nodes, top_k)` runs a BM25Okapi search over a corpus of pre-loaded nodes.
It excels for exact terms and technical codes (e.g. `DDL_SOURCE`, `TB_CUST`) that may lack good semantic embeddings.


In [6]:
from src.retrieval.hybrid_retriever import bm25_search

nodes = [
    {"name": "Customer",        "definition": "A person who buys goods and services",      "synonyms": [],         "column_names": ["CUST_ID", "NAME"],  "node_type": "BusinessConcept"},
    {"name": "Product",         "definition": "A sellable item in the product catalog",    "synonyms": ["Item"],   "column_names": ["PROD_ID", "PRICE"], "node_type": "BusinessConcept"},
    {"name": "Order",           "definition": "A purchase transaction record",             "synonyms": [],         "column_names": ["ORD_ID", "DATE"],   "node_type": "BusinessConcept"},
    {"name": "Invoice",         "definition": "A billing document for a purchase order",   "synonyms": ["Bill"],   "column_names": ["INV_ID", "AMOUNT"], "node_type": "BusinessConcept"},
    {"name": "CustomerAddress", "definition": "Physical address associated with customer", "synonyms": [],         "column_names": ["ADDR_ID", "CITY"],  "node_type": "BusinessConcept"},
]

# Query focalizzata sul customer
results = bm25_search("customer buying goods", nodes, top_k=3)
print("BM25 results for 'customer buying goods':")
for r in results:
    print(f"  [{r.source_type}] {r.node_id} — score={r.score:.4f}")

assert results[0].node_id in ("Customer", "CustomerAddress")
assert all(r.source_type == "bm25" for r in results)
print("✓ bm25_search — top result is most relevant customer-related node")

BM25 results for 'customer buying goods':
  [bm25] Customer — score=1.3950
  [bm25] CustomerAddress — score=0.3606
  [bm25] Product — score=0.0000
✓ bm25_search — top result is most relevant customer-related node


In [7]:
# Empty corpus guard
empty_results = bm25_search("anything", [], top_k=5)
assert empty_results == []
print("✓ bm25_search([]) returns []")

# top_k respected
limited = bm25_search("purchase order", nodes, top_k=2)
assert len(limited) <= 2
print(f"✓ top_k=2 respected: {len(limited)} results")

✓ bm25_search([]) returns []
✓ top_k=2 respected: 2 results


## 4. `vector_search()` — Dense Vector Search

`vector_search(query, client, top_k, model)` performs vector search on Neo4j
via `db.index.vector.queryNodes`. In production it requires a Neo4j index created
with `vector.dimensions: 1024`. Here we mock the Neo4j client.


In [8]:
from unittest.mock import MagicMock
from src.retrieval.hybrid_retriever import vector_search

# Mock Neo4jClient che restituisce record pre-definiti
def _make_client(records):
    client = MagicMock()
    client.execute_cypher.return_value = records
    return client

neo4j_records = [
    {"name": "Customer",  "definition": "Entity representing a buyer", "score": 0.94, "node_type": "BusinessConcept", "source_doc": "glossary.pdf", "synonyms": ["Client"]},
    {"name": "Client",    "definition": "Alternative name for customer", "score": 0.87, "node_type": "BusinessConcept", "source_doc": "glossary.pdf", "synonyms": []},
    {"name": "Purchaser", "definition": "One who makes a purchase",     "score": 0.72, "node_type": "BusinessConcept", "source_doc": "glossary.pdf", "synonyms": []},
]

client = _make_client(neo4j_records)
results = vector_search("who is a customer?", client, top_k=3, model=fake_model)

print("Vector search results:")
for r in results:
    print(f"  [{r.source_type}] {r.node_id} — score={r.score:.2f}")

assert len(results) == 3
assert all(r.source_type == "vector" for r in results)
assert results[0].node_id == "Customer"
print("✓ vector_search — returns RetrievedChunk list with source_type='vector'")

Vector search results:
  [vector] Customer — score=0.94
  [vector] Client — score=0.87
  [vector] Purchaser — score=0.72
✓ vector_search — returns RetrievedChunk list with source_type='vector'


## 5. `graph_traversal()` — Graph Hop Expansion

`graph_traversal(seed_names, client, depth)` expands seed nodes through relations in the Knowledge Graph.
It retrieves neighbours up to `depth` hops away.
Seed nodes themselves are excluded from the results to avoid duplicates.


In [9]:
from src.retrieval.hybrid_retriever import graph_traversal

graph_records = [
    {"name": "Order",   "definition": "Purchase transaction",      "node_type": "BusinessConcept", "rel_type": "PLACED_BY"},
    {"name": "Invoice", "definition": "Billing document",          "node_type": "BusinessConcept", "rel_type": "GENERATED_FROM"},
    {"name": "TB_CUST", "definition": "Physical customer table",   "node_type": "PhysicalTable",   "rel_type": "MAPS_TO"},
]

client = _make_client(graph_records)
results = graph_traversal(["Customer"], client, depth=2)

print(f"Graph neighbours of 'Customer' (depth=2):")
for r in results:
    print(f"  [{r.source_type}] {r.node_id} (rel: {r.metadata.get('rel_type')}) — score={r.score}")

assert all(r.source_type == "graph" for r in results)
assert all(r.node_id != "Customer" for r in results)  # seed excluded
assert all(r.score == 0.5 for r in results)           # neutral baseline
print("✓ graph_traversal — returns neighbours, excludes seeds, score=0.5")

Graph neighbours of 'Customer' (depth=2):
  [graph] Order (rel: PLACED_BY) — score=0.5
  [graph] Invoice (rel: GENERATED_FROM) — score=0.5
  [graph] TB_CUST (rel: MAPS_TO) — score=0.5
✓ graph_traversal — returns neighbours, excludes seeds, score=0.5


In [10]:
# Empty seeds guard
empty = graph_traversal([], _make_client([]))
assert empty == []
print("✓ graph_traversal([]) returns []")

✓ graph_traversal([]) returns []


## 6. `merge_results()` — Multi-Channel Deduplication

`merge_results(vector, bm25, graph)` deduplicates by `node_id`, keeps the maximum
score across channels, and sorts by descending score.


In [11]:
from src.retrieval.hybrid_retriever import merge_results
from src.models.schemas import RetrievedChunk

def _chunk(node_id, score, source="vector"):
    return RetrievedChunk(
        node_id=node_id, node_type="BusinessConcept",
        text=f"{node_id}: definition",
        score=score, source_type=source, metadata={},  # type: ignore
    )

# Simulate overlapping results from three channels
vector_results = [_chunk("Customer", 0.94), _chunk("Client", 0.87)]
bm25_results   = [_chunk("Customer", 0.72, "bm25"), _chunk("Order", 0.65, "bm25")]
graph_results  = [_chunk("Invoice", 0.50, "graph"), _chunk("TB_CUST", 0.50, "graph")]

merged = merge_results(vector_results, bm25_results, graph_results)

print(f"Total unique nodes: {len(merged)}")
print("Merged results (sorted by score desc):")
for r in merged:
    print(f"  {r.node_id:15s} score={r.score:.2f}  source={r.source_type}")

# Customer appears in vector (0.94) and bm25 (0.72) — max is 0.94
customer = next(r for r in merged if r.node_id == "Customer")
assert customer.score == 0.94
assert merged[0].node_id == "Customer"  # highest score first
print("✓ merge_results — deduplication and max-score selection correct")

Total unique nodes: 5
Merged results (sorted by score desc):
  Customer        score=0.94  source=vector
  Client          score=0.87  source=vector
  Order           score=0.65  source=bm25
  Invoice         score=0.50  source=graph
  TB_CUST         score=0.50  source=graph
✓ merge_results — deduplication and max-score selection correct


## 7. `rerank()` — Cross-Encoder Reranking

`rerank(query, chunks, reranker=None, top_k=None)` reorders candidates using a
cross-encoder that reads query and chunk together. Each `(query, chunk.text)` pair
is scored with `reranker.compute_score(pairs, normalize=True)`.

The cross-encoder score overwrites the `.score` field and is also saved in
`metadata["reranker_score"]` for pre/post reranking comparison.


In [12]:
from src.retrieval.reranker import rerank

# Merged pool dal passo precedente (6 candidati)
pool = merged  # from previous cell

# Mock del FlagReranker — ogni coppia riceve un punteggio simulato
reranker_mock = MagicMock()
# Assegna score più alto a Customer e Order
cross_encoder_scores = [0.95, 0.60, 0.88, 0.45, 0.30, 0.20]
reranker_mock.compute_score.return_value = cross_encoder_scores[:len(pool)]

reranked = rerank("customer purchase order", pool, reranker=reranker_mock, top_k=3)

print(f"Pool size: {len(pool)} → Top-K after reranking: {len(reranked)}")
print("\nReranked top-3:")
for r in reranked:
    print(f"  {r.node_id:15s} score={r.score:.4f}  reranker_score={r.metadata.get('reranker_score', 'N/A'):.4f}")

assert len(reranked) <= 3
assert reranked[0].score >= reranked[-1].score  # sorted descending
assert "reranker_score" in reranked[0].metadata
print("\n✓ rerank — top-K selection, score update, metadata preserved")

{"ts": "2026-03-12T16:41:49", "logger": "src.retrieval.reranker", "level": "INFO", "message": "rerank: top chunk 'Customer' score=0.9500 (pool=5, top_k=3)."}


Pool size: 5 → Top-K after reranking: 3

Reranked top-3:
  Customer        score=0.9500  reranker_score=0.9500
  Order           score=0.8800  reranker_score=0.8800
  Client          score=0.6000  reranker_score=0.6000

✓ rerank — top-K selection, score update, metadata preserved


In [13]:
# Graceful degradation on reranker failure
failing_reranker = MagicMock()
failing_reranker.compute_score.side_effect = RuntimeError("Simulated OOM error")

fallback_result = rerank("query", pool[:2], reranker=failing_reranker, top_k=3)
print(f"Fallback returned {len(fallback_result)} chunks (no exception raised)")
assert len(fallback_result) == 2
print("✓ rerank — graceful degradation on failure")

{"ts": "2026-03-12T16:41:49", "logger": "src.retrieval.reranker", "level": "WARNING", "message": "Reranker scoring failed (Simulated OOM error) \u2014 returning chunks unranked."}


Fallback returned 2 chunks (no exception raised)
✓ rerank — graceful degradation on failure


## 8. Full Pipeline — Retrieval End-to-End

Demonstration of the complete retrieval pipeline with all stages chained:
`query → embed → [vector + BM25 + graph] → merge → rerank → top-K chunks`.


In [14]:
# Full pipeline demo (all mocked)
query = "What is the customer entity and which tables map to it?"

# Step 1: embed query
query_vec = embed_text(query, model=fake_model)
print(f"1. Query embedded — dim={len(query_vec)}")

# Step 2: three-channel retrieval (Neo4j mocked)
neo4j_client = _make_client([
    {"name": "Customer",  "definition": "Buyer entity",            "score": 0.91, "node_type": "BusinessConcept", "source_doc": "glossary.pdf", "synonyms": []},
    {"name": "TB_CUST",   "definition": "Physical customer table", "score": 0.78, "node_type": "PhysicalTable",   "source_doc": "ddl.sql",      "synonyms": []},
])
v_results = vector_search(query, neo4j_client, top_k=5, model=fake_model)
b_results = bm25_search(query, nodes, top_k=5)
g_results = graph_traversal(["Customer"], _make_client([
    {"name": "Order", "definition": "Purchase record", "node_type": "BusinessConcept", "rel_type": "PLACED_BY"},
]), depth=2)
print(f"2. Retrieval — vector={len(v_results)}, bm25={len(b_results)}, graph={len(g_results)}")

# Step 3: merge
candidate_pool = merge_results(v_results, b_results, g_results)
print(f"3. Merged pool — {len(candidate_pool)} unique nodes")

# Step 4: rerank
final_reranker = MagicMock()
final_reranker.compute_score.return_value = [0.95 - i * 0.1 for i in range(len(candidate_pool))]
top_chunks = rerank(query, candidate_pool, reranker=final_reranker, top_k=3)
print(f"4. Reranked top-3:")
for chunk in top_chunks:
    print(f"   {chunk.node_id:20s} score={chunk.score:.4f}")

print("\n✓ Full retrieval pipeline executed successfully")

{"ts": "2026-03-12T16:41:49", "logger": "src.retrieval.reranker", "level": "INFO", "message": "rerank: top chunk 'Customer' score=0.9500 (pool=6, top_k=3)."}


1. Query embedded — dim=1024
2. Retrieval — vector=2, bm25=5, graph=1
3. Merged pool — 6 unique nodes
4. Reranked top-3:
   Customer             score=0.9500
   Product              score=0.8500
   TB_CUST              score=0.7500

✓ Full retrieval pipeline executed successfully


## Summary

| Component | Function | Output |
|---|---|---|
| `embed_text(query)` | BGE-M3 1024-dim | `list[float]` |
| `vector_search()` | Semantic similarity via Neo4j | `list[RetrievedChunk]` |
| `bm25_search()` | Exact keyword via BM25Okapi | `list[RetrievedChunk]` |
| `graph_traversal()` | Relational context via Cypher | `list[RetrievedChunk]` |
| `merge_results()` | Dedup by `node_id`, max score | `list[RetrievedChunk]` |
| `rerank()` | Cross-encoder precision | `list[RetrievedChunk]` (top-K) |

The retrieval system implements a classic two-stage pipeline:
1. **Recall stage**: the three channels maximise coverage
2. **Precision stage**: the cross-encoder reorders to maximise relevance

The output is a `list[RetrievedChunk]` ready for the answer generator (Part 8).
